# Data Wrangling

Data for this project is obtained from Inside Airbnb website: http://insideairbnb.com/get-the-data.html. 

Inside Airbnb contains data that is scraped from Airbnb website. Data is scraped at different time points for various locations such as New York, Boston and San Francisco. More data can be found at the archive section as well. 

This dataset contains information for San Francisco and our aim is to create a model that will help airbnb owners to list their units for the optimum price. We will use two csv files in this project:

**listings_detailed.csv:** Detailed information for each listing found on Airbnb website in San Francisco at the time of scraping.

**calendar.csv:**  Pricing information for each listing for the coming year.

In [ ]:
# Importing necessary libraries.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import os
%matplotlib inline

In [ ]:
# Loading the data.
df = pd.read_csv('../data/raw/listings_detailed.csv',index_col='id')
calendar = pd.read_csv('../data/raw/calendar.csv')

In [ ]:
rows1, columns1 = df.shape
rows2, columns2 = calendar.shape

print ('The listing data frame has ', rows1,' rows and ',columns1,' columns.')
print ('The calendar data frame has ', rows2,' rows and ',columns2,' columns.')

In [ ]:
# Getting a glance at all columns of the data frame
pd.set_option('display.max_columns', len(df.columns)) 
df.head(5)

In [ ]:
# printing the summary of the data frame

df.info()

***Dropping the columns***

Columns that contain information such as url's, host details etc will not be used for the initial price prediction. Therefore they will be dropped.

In [ ]:
# Creating a list that has the name of columns that do not contain relevant or 
# useful information or completely null.

columns_to_drop = ['listing_url', 'scrape_id', 'last_scraped', 'name', 'description',
       'neighborhood_overview', 'picture_url', 'host_url',
       'host_name', 'host_location', 'host_about', 'host_thumbnail_url', 'host_picture_url',
       'host_neighbourhood', 'neighbourhood_group_cleansed','bathrooms','calendar_updated','calendar_last_scraped']

# Dropping columns in the list.
df = df.drop(columns_to_drop,axis=1)

Columns that have mostly missing values will be dropped

In [ ]:
# Checking number of missing values in each column and percent as well.
missing = pd.concat([df.isnull().sum(), 100 * df.isnull().mean()], axis=1)
missing.columns=['count', '%']
missing.sort_values(by='count',ascending=False)

In [ ]:
# dropping 'licence' column
df = df.drop('license',axis=1)

**host_listings_count, host_total_listings_count**

Let's check how many rows are there that are different for host_listings_count, host_total_listings_count columns:
A: 15, one of these columns will be dropped.

In [ ]:
# checking the rows that are have different values for 'host_listings_count' and 'host_total_listings_count'
df[df.host_listings_count != df.host_total_listings_count]

**host_listings_count, host_total_listings_count, calculated_host_listings_count**

Let's check calculated_host_listings_count, host_listings_count and, host_total_listings_count columns. What is the difference?

A: 'host_listing_count' and 'host_total_listing_count' columns have different values than the 'calculated_host_listing_count'. Let's group by 'host_id' and check which entry is correct. Based on the host_id = 107434423, there are 177 listings. Thus host_listings_count and host_total_listings_count columns have wrong entries, they will be dropped. 

In [ ]:
# getting a glance at 'id','host_id','host_listings_count','host_total_listings_count','calculated_host_listings_count' columns.
df[['host_id','host_listings_count','host_total_listings_count','calculated_host_listings_count']].sort_values('calculated_host_listings_count',ascending=False)

In [ ]:
# Filtering the data frame using host_id = 107434423
rows, columns = df.query('host_id == 107434423').shape
print('The host with host_id = 107434423 has ', rows, ' listings in total')

In [ ]:
# dropping 'host_listings_count','host_total_listings_count','calculated_host_listings_count_entire_homes' columns

df.drop(['host_listings_count','host_total_listings_count','calculated_host_listings_count_entire_homes'], axis=1,inplace=True)

**minimum_nights,maximum_nights**

There are several columns for minimum and maximum night stays. I will use minimum_nights and maximum_nights columns and drop the rest since there are very few differences between these two main columns and others.

One listing has minimum_nights and maximum_nights set as 10^9. This row will be dropped.

In [ ]:
print(sum((df.minimum_nights == df.minimum_minimum_nights) == False))
print(sum((df.minimum_nights == df.maximum_minimum_nights) == False))
print(sum((df.maximum_nights == df.minimum_maximum_nights) == False))
print(sum((df.maximum_nights == df.maximum_maximum_nights) == False))
print(sum((df.maximum_nights == df.maximum_nights_avg_ntm) == False))
print(sum((df.minimum_nights == df.minimum_nights_avg_ntm) == False))

In [ ]:
# dropping 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights','minimum_nights_avg_ntm','maximum_nights_avg_ntm' columns 
df.drop(['minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights','minimum_nights_avg_ntm','maximum_nights_avg_ntm'],axis=1,inplace=True)

In [ ]:
print(df[['minimum_nights']].max())
print(df[['maximum_nights']].max())

In [ ]:
# dropping the row that has minimum_nights and maximum_nights set as 10^9.
df = df[df.minimum_nights != df.minimum_nights.max()]

**changing entries coded as 't' or 'f' with 1 or 0**

Some of the columns contained entries as t/f referring to t:true and f:false. These columns will be converted to 1 and 0.

In [ ]:
# Replacing columns with f/t with 0/1
df.replace({'f': 0, 't': 1}, inplace=True)

**Cleaning Individual Columns:**

**host_since**

This column is recognized as object type but it is a date time column. First I will convert it to date time then I will create a new column as host_days_active by subtracting host_since from the data scraping date: 2021-03-02. I will use median value to fill the missing values.

In [ ]:
# Converting to datetime
df.host_since = pd.to_datetime(df.host_since) 

# Calculating the number of days
df['host_days_active'] = (datetime(2021, 3, 2) - df.host_since).astype('timedelta64[D]')

# Printing mean and median
print("Mean days as host:", round(df['host_days_active'].mean(),0))
print("Median days as host:", df['host_days_active'].median())

# Replacing null values with the median
df.host_days_active.fillna(df.host_days_active.median(), inplace=True)

**host_response_time**

I will encode missing values as unknown and use dummies encoding later on for this column.

In [ ]:
# Checking the unique values in the host_response_time column
df.host_response_time.unique()

In [ ]:
print('Proportion: ', round(df.host_response_time.isna().sum()/len(df),2))
print('There are: ',df.host_response_time.isna().sum(), 'missing values.')

In [ ]:
# replacing missing values with unknown
df.host_response_time.fillna("unknown", inplace=True)

**host_response_rate**

The entries in this column are strings as: '%95', '%100', etc. I will remove the % and convert the string to numeric type.

In [ ]:
# removing the % and converting string to numeric values
df['host_response_rate'] = pd.to_numeric(df['host_response_rate'].str.replace('%',''))
print('Proportion: ', round(df.host_response_rate.isna().sum()/len(df),2))
print('There are: ',df.host_response_rate.isna().sum(), 'missing values.')

In [ ]:
print('mean: ', df.host_response_rate.mean())
print('median: ', df.host_response_rate.median())
print('ratio of 100%s: ', (df.host_response_rate==100).sum()/len(df))

In [ ]:
# Bin into four categories
df.host_response_rate = pd.cut(df.host_response_rate, bins=[0, 50, 90, 99, 100], labels=['0-49%', '50-89%', '90-99%', '100%'], include_lowest=True)

# Converting to string
df.host_response_rate = df.host_response_rate.astype('str')

# Replace nulls with 'unknown'
df.host_response_rate.fillna('unknown', inplace=True)

# Category counts
df.host_response_rate.value_counts()

**host_acceptance_rate**

The entries in this column are strings as: '%95', '%100', etc. I will remove the % and convert the string to numeric type. 

In [ ]:
df['host_acceptance_rate'] = pd.to_numeric(df['host_acceptance_rate'].str.replace('%',''))
print('Proportion: ', round(df.host_acceptance_rate.isna().sum()/len(df),2))
print('There are: ',df.host_acceptance_rate.isna().sum(), 'missing values.')

In [ ]:
print('mean: ', df.host_acceptance_rate.mean())
print('median: ', df.host_acceptance_rate.median())
print('ratio of 100%s: ', (df.host_acceptance_rate==100).sum()/len(df))

In [ ]:
# Bin into four categories
df.host_acceptance_rate = pd.cut(df.host_acceptance_rate, bins=[0, 50, 90, 99, 100], labels=['0-49%', '50-89%', '90-99%', '100%'], include_lowest=True)

# Converting to string
df.host_acceptance_rate = df.host_acceptance_rate.astype('str')

# Replace nulls with 'unknown'
df.host_acceptance_rate.fillna('unknown', inplace=True)

# Category counts
df.host_acceptance_rate.value_counts()

**host_is_superhost**

This column has 1,0's as entries and has only 15 missing values. I will fill the missing values using median.

In [ ]:
print('host_is_superhost column has: ',df.host_is_superhost.isna().sum(),' missing values.')
df.host_is_superhost.fillna(df.host_is_superhost.median(),inplace=True)

**host_verifications**

There are number of methods for host verification, i.e email, facebook, google, government_id. I can categorize these methods as follows: government_id based, other, and none. I will replace the columns that contain government_id with 2, none with 0 and others with 1.

In [ ]:
# creating a copy of df.host_verifications
df1 = df.host_verifications.copy()
# removing the '[',']',"'",' ' characters from the string
char_list = ['[',']',"'",' ']
for i in char_list:
    df1 = df1.str.replace(i,'',regex=True)

# splitting the string based on ',' as separator.
df1 = df1.str.split(',')

In [ ]:
# creating a list from df1 dataframe
ver_list = df1.values.tolist()
ver_list_1 = []
# creating a list using all values of the lists in the df1 dataframe
for i in ver_list:
    for j in i:
        ver_list_1.append(j)
# creating a set of all entries in host_verification column
set(ver_list_1)

In [ ]:
df.loc[df.host_verifications.str.contains('None'),'host_verifications'] = '0'
df.loc[df.host_verifications.str.contains('email|facebook|google|jumio|kba|manual_offline|manual_online|phone|reviews|selfie|sent_id|work_email|zhima_selfie'),'host_verifications'] = '1'
df.loc[df.host_verifications.str.contains('government'),'host_verifications'] = '2'
df.host_verifications = pd.to_numeric(df.host_verifications)

**host_has_profile_pic, host_identity_verified**

These two columns have 1 or 0 as entries and both have 15 missing values. Missing values will be replace with median values for both columns.

In [ ]:
print('host_has_profile_pic column has: ',df.host_has_profile_pic.isna().sum(),' missing values.')
df.host_has_profile_pic.fillna(df.host_has_profile_pic.median(),inplace=True)

print('host_identity_verified column has: ',df.host_identity_verified.isna().sum(),' missing values.')
df.host_identity_verified.fillna(df.host_identity_verified.median(),inplace=True)


**neighbourhood, neighbourhood_cleansed**

neighbourhood column contains cities and it is too broad. neighbourhood_cleansed column has more granular neighbourhood information. neighbourhood column will be dropped and neighbourhood_cleansed column will be renamed as neighbourhood.

In [ ]:
# lets check the neighbourhood column
df.neighbourhood.unique()

In [ ]:
# lets check the neighbourhood_cleansed column
df.neighbourhood_cleansed.unique()

In [ ]:
#Dropping the neighbourhood column
df = df.drop('neighbourhood',axis=1)
df.rename(columns={'neighbourhood_cleansed':'neighbourhood'},inplace=True)

**latitude, longitude**

Both of these columns will be assigned to different data frame. I will use them to calculate distance to landmarks in San Francisco

In [ ]:
lat_long = df[['latitude', 'longitude']]
df.drop(['latitude', 'longitude'],axis=1,inplace=True)

**room_type, property_type**

Entries in property_type column are sub-groups of entries in room_type column. This column will be dropped. There are 4 unique entries in room_type. room_type column will be renamed as property_type.

In [ ]:
df['room_type'].unique()

In [ ]:
df['property_type'].unique()

In [ ]:
df.drop('property_type',axis=1,inplace=True)
df.rename(columns={'room_type':'property_type'},inplace=True)

**bathroom_text**

bathrooms columns were missing values completely and dropped previously. bathrooms_text column contains strings such as '1 bath', '1 private bath', '1 shared bath'. Are the entries as ' private bath' and ' shared bath' specific to room_types as 'Private room' and 'Shared room', respectiively? If so I can remove the letters and convert the entries to numeric values.

Yes, Private rooms have -private bath and Shared rooms have -shared bath as entries. 

In [ ]:
# Bathrooms column was completely null but there is another column 'bathrooms_text'
# Let's look at the unique values in the 'bathroom_text' column.

df['bathrooms_text'].value_counts()

In [ ]:
property_type = df.property_type.unique()
[print(i, ' has following bathrooms:\n', df[df['property_type']==i]['bathrooms_text'].unique(),'\n') for i in property_type];

In [ ]:
# Replacing '...half-bathooms' with 0.5 and removing the letters from the remaining rows leaving only numbers.
df['bathrooms_text'] = df['bathrooms_text'].replace('Half-bath','0.5').replace('Shared half-bath','0.5').replace('Private half-bath','0.5').str.replace(' shared baths','').str.replace(' shared bath','').str.replace(' private bath','').str.replace(' baths','').str.replace(' bath','')
# Changing data type of bathrooms_text column to numeric.
df['bathrooms_text'] = pd.to_numeric(df['bathrooms_text'])
# Renaming bathrooms_text column to bathrooms.
df.rename(columns={'bathrooms_text':'bathrooms'},inplace=True)
df.bathrooms.fillna(df.bathrooms.median(),inplace=True)

**accommodates,bedrooms, beds**

bedroom and beds columns has missing values. Missing values are filled using median.

In [ ]:
print('accomodates column has: ', df.accommodates.isna().sum(),' missing values.')
print('bedrooms column has: ', df.bedrooms.isna().sum(),' missing values.')
print('beds column has: ', df.beds.isna().sum(),' missing values.')

In [ ]:
df.bedrooms.fillna(df.bedrooms.median(),inplace=True)
df.beds.fillna(df.beds.median(),inplace=True)

**price**

Entries in this column are string type. $ and , characters are removed and entries are converted to numeric type

In [ ]:
# Price column type is object. Contains $ sign and comma in some rows, 
# so I will remove both and convert the remaining string to numeric.

df['price'] = df['price'].str.replace(',','',regex=True).str.replace('$','',regex=True)
df['price'] = pd.to_numeric(df['price'])

**amenities**

This column has a list of amenities present in the airbnb unit. Entries are string type so first it will be converted to a list. 

In [ ]:
# removing the '[',']',"'",' ' characters from the string
char_list = ['[',']','"']
df1.amenities = df.amenities.copy()
for i in char_list:
    df1.amenities = df1.amenities.str.replace(i,'',regex=True)

# splitting the string based on ',' as separator.
df1.amenities = df1.amenities.str.split(',')

In [ ]:
# creating a list from df1 dataframe
amenities_list = df1.amenities.values.tolist()
amenities_list_1 = []
# creating a list using all values of the lists in the df1 dataframe
for i in amenities_list:
    for j in i:
        amenities_list_1.append(j)
# creating a set of all entries in host_verification column
set(amenities_list_1)

In [ ]:
# making all entries lowercase
df.amenities = df.amenities.str.lower()

The amenities chosen are (slashes indicate separate categories that can be combined):
HD/TVwithAmazonPrime/Netflix (i.e. non-basic electronics)
Cable TV/TV


Air conditioning/central air conditioning / Portableairconditioning
'BBQgrill'/'Firepit'/'Naturalgasbarbeque'
'Balcony'/'Patioorbalcony'/
'Bedlinens'/
'Breakfast'
'Coffee'/'Espressomachine'
'Cookingbasics'
'Elevator'
'gym'
anything containing child
anything containing parking
'Hostgreetsyou'
'Longtermstaysallowed'
'Petsallowed'
'Suitableforevents'
'Smokingallowed'
'Selfcheck-in'
'Privateentrance'
'wifi'
'Hottub'/'Jettedtub'/sauna/pool
garden/backyard/outdoor/terrace/loung
'Dishwasher'/dryer/'washer'
'safe'/'Securitycameras'
'Beachfront'/'Lakeaccess'/ 'Ski-in/Ski-out'/'Waterfront'




















In [ ]:
df.loc[df['amenities'].str.contains('tv'), 'tv'] = 1
df.loc[df['amenities'].str.contains('amazon echo|apple tv|console|netflix|smart tv|tv with'), 'high_end_electronics'] = 1
df.loc[df['amenities'].str.contains('air conditioning'), 'air_conditioning'] = 1
df.loc[df['amenities'].str.contains('bbq grill|fire pit|natural gas barbeque'), 'bbq'] = 1
df.loc[df['amenities'].str.contains('balcony|patio or balcony'), 'balcony'] = 1
df.loc[df['amenities'].str.contains('bed linens'), 'bed_linen'] = 1
df.loc[df['amenities'].str.contains('breakfast'), 'breakfast'] = 1
df.loc[df['amenities'].str.contains('coffee|espresso machine'), 'coffee_machine'] = 1
df.loc[df['amenities'].str.contains('cooking basics'), 'cooking_basics'] = 1
df.loc[df['amenities'].str.contains('elevator'), 'elevator'] = 1
df.loc[df['amenities'].str.contains('gym'), 'gym'] = 1
df.loc[df['amenities'].str.contains('child'), 'child_friendly'] = 1
df.loc[df['amenities'].str.contains('parking'), 'parking'] = 1
df.loc[df['amenities'].str.contains('host greets you'), 'host_greeting'] = 1
df.loc[df['amenities'].str.contains('long term stays allowed'), 'long_term_stays'] = 1
df.loc[df['amenities'].str.contains('pets allowed'), 'pets_allowed'] = 1
df.loc[df['amenities'].str.contains('suitable for events'), 'event_suitable'] = 1
df.loc[df['amenities'].str.contains('smoking allowed'), 'smoking_allowed'] = 1
df.loc[df['amenities'].str.contains('self check-in'), 'self_check_in'] = 1
df.loc[df['amenities'].str.contains('private entrance'), 'private_entrance'] = 1
df.loc[df['amenities'].str.contains('wifi'), 'internet'] = 1
df.loc[df['amenities'].str.contains('hot tub|jetted tub|sauna|pool'), 'hot_tub_sauna_or_pool'] = 1
df.loc[df['amenities'].str.contains('garden|backyard|outdoor|terrace|lounge'), 'outdoor_space'] = 1
df.loc[df['amenities'].str.contains('dishwasher|dryer|washer'), 'white_goods'] = 1
df.loc[df['amenities'].str.contains('safe|security cameras'), 'secure'] = 1
df.loc[df['amenities'].str.contains('beachfront|lake access|ski-in/ski-out|waterfront'), 'nature_and_views'] = 1


In [ ]:
df.loc[:,'tv':'nature_and_views'].info()

In [ ]:
cols_to_replace_nulls = df.loc[:,'tv':'nature_and_views'].columns

In [ ]:
# Replacing nulls with zeros for new columns
cols_to_replace_nulls = df.loc[:,'tv':'nature_and_views'].columns
df[cols_to_replace_nulls] = df[cols_to_replace_nulls].fillna(0)

# Produces a list of amenity features where one category (true or false) contains fewer than 10% of listings
infrequent_amenities = []
for col in cols_to_replace_nulls:
    if df[col].sum() < len(df)/10:
        infrequent_amenities.append(col)
print(infrequent_amenities)

# Dropping infrequent amenity features
df.drop(infrequent_amenities, axis=1, inplace=True)

# Dropping the original amenity feature
df.drop('amenities', axis=1, inplace=True)

**has_availability,availability_30,availability_60,availability_90,availability_365**

has_availability column tells us whether the listing is available or not at the date of data scraping and availability_30,60,90 and _365 tells us the availability of the listings in those time periods. has_availability will be dropped. availability_30,_90,_365 will be dropped since they have high correlation between them and _60 will be kept.

In [ ]:
df.drop(['has_availability','availability_30','availability_90','availability_365'],axis=1, inplace=True)

**number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d**

number_of_reviews column has the total number of reviews per listing. 
number_of_reviews_ltm has the reviews given in last 12-months.
number_of_reviews_l30d has the reviews given in last 30-days.

number_of_reviews_ltm,number_of_reviews_l30d has high correlation with num_of_reviews thus they will be dropped.

In [ ]:
df[['number_of_reviews','number_of_reviews_ltm','number_of_reviews_l30d','reviews_per_month']].corr()

In [ ]:
df.drop(['number_of_reviews_ltm','number_of_reviews_l30d','reviews_per_month'], axis=1,inplace=True)

**first_review,last_review**

In [ ]:
# Converting to datetime
df.first_review = pd.to_datetime(df.first_review) 
df.last_review = pd.to_datetime(df.last_review) 

In [ ]:
print('The ratio of missing values in first_review column is: ', round(df.first_review.isna().sum()/df.first_review.shape[0],2))
print('The ratio of missing values in last_review column is: ', round(df.last_review.isna().sum()/df.last_review.shape[0],2))

In [ ]:
df['since_first_review'] = (datetime(2021, 3, 2) - df.first_review).astype('timedelta64[D]')
df['since_last_review'] = (datetime(2021, 3, 2) - df.last_review).astype('timedelta64[D]')

In [ ]:
# distribution of since_first_review and since_last_review columns:

plt.subplot(2,1,1)
df.since_first_review.hist(figsize=(10,5), bins=50)
plt.subplot(2,1,2)
df.since_last_review.hist(figsize=(10,5), bins=50)
plt.show()

In [ ]:
# creating bins for the since_first_review column
df.since_first_review = pd.cut(df.since_first_review, 
                               bins = [0,180,365,730,1460,max(df.since_first_review)], 
                               labels = ['0-6 months',
                                         '6-12 months',
                                         '1-2 years',
                                         '2-3 years',
                                         '4+ years'], include_lowest=True)

df.since_first_review = df.since_first_review.astype('str')
df.since_first_review.fillna('no_reviews',inplace=True)

In [ ]:
# creating bins for the since_last_review column

df.since_last_review = pd.cut(df.since_last_review, 
                               bins = [0,14,60,180,365,max(df.since_last_review)], 
                               labels = ['0-2 weeks',
                                         '2-8 weeks',
                                         '2-6 months',
                                         '6-12 months',
                                         '1+ year'], include_lowest=True)

df.since_last_review = df.since_last_review.astype('str')
df.since_last_review.fillna('no_reviews',inplace=True)

In [ ]:
#dropping the first_review and last_review columns
df.drop(['first_review','last_review'],axis=1,inplace=True)

**reviews**

Missing values in each reviews column is replaced with its median.
         

In [ ]:

# Checking the distributions of the review ratings columns
variables_to_plot = list(df.columns[df.columns.str.startswith("review_scores") == True])
fig = plt.figure(figsize=(12,8))
for i, var_name in enumerate(variables_to_plot):
    ax = fig.add_subplot(3,3,i+1)
    df[var_name].hist(bins=10,ax=ax)
    ax.set_title(var_name)
fig.tight_layout()
plt.show()

In [ ]:
# Bin into four categories
cols = ['review_scores_accuracy','review_scores_cleanliness','review_scores_checkin','review_scores_communication','review_scores_location','review_scores_value']

 
for i in cols:
    df[i] = pd.cut(df[i], bins=[0, 8, 9, 10], labels=['0-8', '9', '10'], include_lowest = True)
    df[i] = df[i].astype('str')
    df[i].fillna('no_reviews',inplace=True)

In [ ]:
df.review_scores_rating = pd.cut(df.review_scores_rating, 
                               bins = [0, 80, 95, 100], 
                               labels = ['0-79', '80-94', '95-100'], include_lowest=True)
df.review_scores_rating = df.review_scores_rating.astype('str')
df.review_scores_rating.fillna('no_reviews',inplace=True)


In [ ]:
# Getting a glance at the calendar data frame
calendar.head()

In [ ]:
# Looking at the data types of the columns.
calendar.info()

In [ ]:
# Changing data type of date column to datetime
calendar['date'] = pd.to_datetime(calendar['date'])

In [ ]:
# Checking the unique values in the available column.
calendar.available.unique()

In [ ]:
# Mapping f and t in the available column to 0 and 1, respectively.
calendar.available = calendar.available.replace({'f':0,'t':1})

In [ ]:
# Checking the unique values in the available column.
calendar.available.unique()

In [ ]:
# Removing the '$' sign and ',' from the price and adjusted price columns and convert them to numeric type.
calendar['price'] = calendar['price'].str.replace(',','').str.replace('$','')
calendar['adjusted_price'] = calendar['adjusted_price'].str.replace(',','').str.replace('$','')
calendar.price = pd.to_numeric(calendar.price)
calendar.adjusted_price = pd.to_numeric(calendar.adjusted_price)

In [ ]:
# Checking the data types of the columns
calendar.info()

In [ ]:
# Saving the cleaned data into csv file.
calendar.to_csv(r'../data/clean/calendar_clean.csv', index = False, header=True)

In [ ]:
# Saving the cleaned data into csv file.
df.to_csv(r'../data/clean/listings_clean_dummy_encoding.csv', index = False, header=True)